In [ ]:
text = """
Cybersecurity is an important field of technology.
It protects computers, networks, and digital information.
Cybersecurity helps prevent unauthorized access to data.
Hackers try to steal sensitive information from users.
Phishing attacks trick people into sharing passwords.
Malware is harmful software that can damage computers.
Strong passwords help protect online accounts.
Two-factor authentication provides an extra layer of security.
Firewalls help protect networks from unauthorized access.
Encryption keeps sensitive information secure.
Antivirus software detects and removes harmful programs.
Cybersecurity protects personal and financial information.
Regular software updates help fix security vulnerabilities.
Public Wi-Fi networks can create security risks.
Users should avoid clicking on suspicious links.
Data privacy is an important part of cybersecurity.
Ethical hackers identify security weaknesses.
Network security protects systems from cyber threats.
Organizations use security tools to protect their data.
Cyberattacks can cause serious financial losses.
Cybersecurity professionals monitor suspicious activities.
Artificial intelligence is improving cybersecurity systems.
Learning cybersecurity creates new career opportunities.
Every internet user should follow online safety practices.
Cybersecurity helps create a safer digital world.
"""

In [ ]:
text = text.lower()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
#tokenization
words = text.split()

In [ ]:
# remove the repetition from the dataset
vocab = sorted(set(word))

In [ ]:
vocab_size = len(vocab)
print(vocab_size)

121


In [ ]:
word_to_id = {
    word : i
    for word, i in enumerate(vocab)
}


In [ ]:
print(word_to_id)

{0: 'a', 1: 'access', 2: 'access.', 3: 'accounts.', 4: 'activities.', 5: 'an', 6: 'and', 7: 'antivirus', 8: 'artificial', 9: 'attacks', 10: 'authentication', 11: 'avoid', 12: 'can', 13: 'career', 14: 'cause', 15: 'clicking', 16: 'computers,', 17: 'computers.', 18: 'create', 19: 'creates', 20: 'cyber', 21: 'cyberattacks', 22: 'cybersecurity', 23: 'cybersecurity.', 24: 'damage', 25: 'data', 26: 'data.', 27: 'detects', 28: 'digital', 29: 'encryption', 30: 'ethical', 31: 'every', 32: 'extra', 33: 'field', 34: 'financial', 35: 'firewalls', 36: 'fix', 37: 'follow', 38: 'from', 39: 'hackers', 40: 'harmful', 41: 'help', 42: 'helps', 43: 'identify', 44: 'important', 45: 'improving', 46: 'information', 47: 'information.', 48: 'intelligence', 49: 'internet', 50: 'into', 51: 'is', 52: 'it', 53: 'keeps', 54: 'layer', 55: 'learning', 56: 'links.', 57: 'losses.', 58: 'malware', 59: 'monitor', 60: 'network', 61: 'networks', 62: 'networks,', 63: 'new', 64: 'of', 65: 'on', 66: 'online', 67: 'opportuniti

In [ ]:
id_to_word = {
    i : word
    for word, i in word_to_id.items()
}

In [ ]:
print(id_to_word)

{'a': 0, 'access': 1, 'access.': 2, 'accounts.': 3, 'activities.': 4, 'an': 5, 'and': 6, 'antivirus': 7, 'artificial': 8, 'attacks': 9, 'authentication': 10, 'avoid': 11, 'can': 12, 'career': 13, 'cause': 14, 'clicking': 15, 'computers,': 16, 'computers.': 17, 'create': 18, 'creates': 19, 'cyber': 20, 'cyberattacks': 21, 'cybersecurity': 22, 'cybersecurity.': 23, 'damage': 24, 'data': 25, 'data.': 26, 'detects': 27, 'digital': 28, 'encryption': 29, 'ethical': 30, 'every': 31, 'extra': 32, 'field': 33, 'financial': 34, 'firewalls': 35, 'fix': 36, 'follow': 37, 'from': 38, 'hackers': 39, 'harmful': 40, 'help': 41, 'helps': 42, 'identify': 43, 'important': 44, 'improving': 45, 'information': 46, 'information.': 47, 'intelligence': 48, 'internet': 49, 'into': 50, 'is': 51, 'it': 52, 'keeps': 53, 'layer': 54, 'learning': 55, 'links.': 56, 'losses.': 57, 'malware': 58, 'monitor': 59, 'network': 60, 'networks': 61, 'networks,': 62, 'new': 63, 'of': 64, 'on': 65, 'online': 66, 'opportunities.'

In [ ]:
data = torch.tensor([id_to_word[word] for word in words], dtype=torch.long)
print(type(data))

<class 'torch.Tensor'>


In [ ]:
#parameters
batch_size = 4
embedding_dim = 32

In [ ]:
#input and target
def get_batch():
  start = torch.randint(
      len(data) - batch_size,(4,)
  )

  x = torch.stack([data[i: i+batch_size] for i in start])
  y = torch.stack([data[i+1: i+batch_size+1] for i in start])

  return x, y

In [ ]:
class MiniModel(nn.Module):
  def __init__(self, vocab_size, embedding_dim, max_len=100):
    super().__init__()

    #token embedding
    self.embedding = nn.Embedding(vocab_size, embedding_dim)

    #positional embedding
    self.position_embedding = nn.Embedding(max_len, embedding_dim)

    #transformer layer
    self.transformer = nn.TransformerEncoderLayer(
        nhead = 4,
        d_model=embedding_dim,
        batch_first=True
    )

    #output layer
    self.linear = nn.Linear(embedding_dim, vocab_size)

  def forward(self, x):
    B, T = x.shape

    tokens = self.embedding(x)

    # Get positions on the correct device
    positions = torch.arange(T, device=x.device)
    position = self.position_embedding(positions)

    x = tokens + position

    # causal mask
    mask = torch.triu(
        torch.ones(T, T, device=x.device),
        diagonal=1
    ).bool() # transformer layers expect boolean or float masks

    x = self.transformer(x, src_mask=mask)

    output = self.linear(x)

    return output

In [ ]:
model = MiniModel(vocab_size, embedding_dim)

#optimize
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
#training
for epoches in range(1000):
  x,y = get_batch()

  prediction = model(x)

  loss = F.cross_entropy(prediction.view(-1, vocab_size), y.view(-1))

  optimizer.zero_grad()

  loss.backward()

  optimizer.step()

  if epoches % 100 == 0:
    print(f"loss: {loss.item()}")

loss: 4.8950629234313965
loss: 3.2096266746520996
loss: 2.392955780029297
loss: 1.5890799760818481
loss: 0.5809951424598694
loss: 0.6444857120513916
loss: 0.3459852933883667
loss: 0.5887857675552368
loss: 0.3442522883415222
loss: 0.2264338582754135


In [ ]:
model.eval()

MiniModel(
  (embedding): Embedding(121, 32)
  (position_embedding): Embedding(100, 32)
  (transformer): TransformerEncoderLayer(
    (self_attn): MultiheadAttention(
      (out_proj): NonDynamicallyQuantizableLinear(in_features=32, out_features=32, bias=True)
    )
    (linear1): Linear(in_features=32, out_features=2048, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (linear2): Linear(in_features=2048, out_features=32, bias=True)
    (norm1): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
    (norm2): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
    (dropout1): Dropout(p=0.1, inplace=False)
    (dropout2): Dropout(p=0.1, inplace=False)
  )
  (linear): Linear(in_features=32, out_features=121, bias=True)
)

In [ ]:
user_input = input("Enter a word: ").lower()

context = torch.tensor([[id_to_word[user_input]]])

with torch.no_grad():
  for i in range(10):
    prediction = model(context[:, -batch_size:])

    prediction = prediction[:, -1, :]

    probabilities = F.softmax(prediction, dim=1)

    next_word = torch.multinomial(probabilities, num_samples=1)

    context = torch.cat([context, next_word], dim=1)

generated_text = " ".join(
    word_to_id[i] for i in context[0].tolist()
)

print(generated_text)

Enter a word: Cybersecurity
cybersecurity creates new career opportunities. every internet user should avoid updates
